In [80]:
import tiktoken
from torch.utils.data import Dataset,DataLoader
import torch

In [81]:
### initing random ids
input_token_ids=torch.tensor([2,3,5])

In [82]:
vocabulary_size=10
n_dim=6 ## dimention of embedding

In [83]:
embedding=torch.nn.Embedding(vocabulary_size,n_dim)
embedding.weight.size()

torch.Size([10, 6])

In [84]:
## lookup table for 10 words
embedding.weight

Parameter containing:
tensor([[-1.8129,  0.3494, -1.1577, -1.9413, -0.6970, -1.7139],
        [ 0.3781, -0.3347, -0.1092, -0.5623,  0.3184, -1.9285],
        [ 0.6065,  0.4754, -0.7882,  1.4592,  2.0834,  0.9105],
        [ 0.0087, -0.6360, -0.7846,  0.5140, -0.1798, -0.2789],
        [-0.0756, -0.8026, -1.4611, -0.2108,  2.3383,  1.1697],
        [ 0.6353,  2.0356, -1.0705, -0.2023, -0.9944,  1.2912],
        [ 1.1070, -0.0432, -1.4610, -0.1867, -1.0604, -0.8569],
        [-0.0630, -0.7053,  0.2133,  0.6479,  1.2535,  0.9143],
        [-1.4630, -0.5082,  0.4312,  1.4756, -1.5551,  0.0061],
        [ 1.5445,  1.3866, -0.2649, -1.1607,  0.8797,  0.7897]],
       requires_grad=True)

In [85]:
embedding.weight[input_token_ids]

tensor([[ 0.6065,  0.4754, -0.7882,  1.4592,  2.0834,  0.9105],
        [ 0.0087, -0.6360, -0.7846,  0.5140, -0.1798, -0.2789],
        [ 0.6353,  2.0356, -1.0705, -0.2023, -0.9944,  1.2912]],
       grad_fn=<IndexBackward0>)

In [86]:
## lets make it practical
vocabulary_size=50257 ## max limit for tokenizer for gpt2
n_dim=512
embedding_layer=torch.nn.Embedding(vocabulary_size,n_dim)

In [87]:
class CustomDataset(Dataset):
    def __init__(self,raw_text,context_size,tokenizer,strides):
        self.input_ids=[]
        self.output_ids=[]
        tokens=tokenizer.encode(raw_text,allowed_special={"<|endoftext|>"})
        for i in range(0,len(tokens)-context_size,strides):
            input_chunk=tokens[i:i+context_size]
            output_chunk=tokens[i+1:i+1+context_size]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))

    def __len__(self):
        return len(self.input_ids)    
    def __getitem__(self, index):
        return self.input_ids[index],self.output_ids[index]    
        

In [88]:
with open("the-verdict.txt","r") as file:
    text=file.read()

In [89]:
def create_dataloader(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
   ## here max_length=context_size

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = CustomDataset(txt,context_size=max_length,tokenizer=tokenizer, strides=stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [90]:
max_length=4 ## context window size is 4
data_loader=create_dataloader(text,batch_size=8,max_length=max_length,stride=max_length)


In [91]:
data=iter(data_loader)
x,y=next(data)

In [92]:
x

tensor([[   11,   355,  4544,  9325],
        [  339,  2993,   655,   644],
        [  286,   428,  3991,  4118],
        [ 5986,    30,  2011, 20136],
        [  262,  6678, 40315, 10455],
        [  338,   281, 12659,  2829],
        [   62,   410,  1386, 20394],
        [   84, 16579,    13,   383]])

In [93]:
token_embedding=embedding_layer(x)

In [94]:
embedding_layer(x).shape

torch.Size([8, 4, 512])

In [95]:
### for positional encoding
positional_encoding=torch.nn.Embedding(max_length,n_dim)

In [96]:
positional_embedding=positional_encoding(torch.arange(max_length)) ## positional encoding for entire batch is same

In [ ]:
## for first batch we have input x
input_embedding=token_embedding+positional_embedding ## adding positional embedding to each token embedding

In [99]:
input_embedding.size()

torch.Size([8, 4, 512])